## 1.  Pivot()
## 
**pivot() rotates unique values from a single categorical column into multiple distinct output columns, aggregating data across each row group.\

It requires a groupBy() beforehand and an aggregation function afterwards.**

In [0]:
data = [
    ("Bhupesh", "Q1", 5000),
    ("Bhupesh", "Q2", 6000),
    ("Shivani", "Q1", 7000),
    ("Shivani", "Q2", 8000),
    ("Rani", "Q1", 4000),
]
schema = ["name", "quarter", "sales"]

df = spark.createDataFrame(data, schema)
df.show()


from pyspark.sql.functions import sum

pivoted_df = df.groupBy("name").pivot("quarter").sum("sales")

pivoted_df.show()

## 2. unpivot()
## 

**Rotating column into rows , Pyspark sql doesn't have unpivot() Fuction so it use stack() Function.**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr

data = [('IT', 20, 15), ('HR', 7, 8), ('payroll', 3, 5)]
schema = ['department', 'male', 'female']

df = spark.createDataFrame(data, schema)
df.show()

# Entire SQL expression enclosed within string quotes
unpivot_df = df.select(
    'department',
    expr("stack(2, 'Male', male, 'Female', female) as (gender, count)")
)

unpivot_df.show()

## 3. fillna() fill()
## 

### replacing with some values at null position

In [0]:
data = [(1 , 'Bhupesh' , 23 ,None , 30000) , (2 , 'Shivani' , 23 ,'Female' , 30000) , (3 , 'Geeta' , 30 ,'Female' , None)]
schema = ['id' , 'name' , 'age' , 'gender' , 'salary']

df = spark.createDataFrame(data, schema)
df.show()



# Now we will fill the null values with diffrent methods

scenerio 1

In [0]:
# Its not filling at salary column  because datatype is diffret so , we can seperately fill values by column

df.fillna('unknown').show()      


 # we can use na.fill() also  ,  and in () parameter here we gave specific column name in which we have to fill the null values

df.na.fill('unknown' , ['gender' , 'salary']).show()    

scenerio 2

In [0]:
df.fillna({'gender':'unknown' , 'salary' :0}).show()    # here we giving we every specific column name and value which we have to fill

## 4. sample() 
## 
**Returns a random sample subset of rows from a DataFrame.\
It is commonly used for exploratory data analysis, testing pipelines on smaller datasets, or splitting data into training and testing sets.**

### scenerio 1

in every sample it contains diffrent value even percent ratio same 

In [0]:
df = spark.range(start = 0 , end=100)
display(df)

df1= df.sample(fraction = 0.1)
display(df1)

df2=df.sample(fraction=0.1)
display(df2)


 # even df1 and df2 are same but in output sample rows comes diffrent

### # scenerio 2 
### 
to protect the sample again and again usee seed parameter and give a value in both sample

In [0]:
df = spark.range(start = 0 , end=100)


df1= df.sample(fraction = 0.1 , seed =120)
display(df1)

df2=df.sample(fraction=0.1 , seed=120)
display(df2)

### scenrio 3 

withReplacement \

False (default): Rows are sampled without replacement (a row can only appear once in the sample).\

True: Rows are sampled with replacement (a single row can be selected multiple times).

In [0]:
df = spark.range(start = 0 , end=100)


df1= df.sample(fraction = 0.1 , seed =120 , withReplacement=False)
display(df1)

df2=df.sample(fraction=0.1 , seed=120 , withReplacement=False)
display(df2)

## 5. collect()
## 
**collect() retrieves all rows from a distributed PySpark DataFrame and returns them to the driver node as a Python list of Row objects.**

### The main motive of collect() is to bridge the gap between distributed PySpark and standard Python.\
### 
### PySpark DataFrames live across multiple worker nodes in a cluster. Regular Python libraries (like Pandas, Scikit-Learn, Matplotlib, or standard Python for loops) cannot process data while it is distributed. collect() pulls that data into driver memory so you can use standard Python tools.

In [0]:
data = [(1, 'Bhupesh' , 23) , (2, 'Shivani' ,24)]
schema = ['id', 'name', 'age']

df =spark.createDataFrame(data,schema)
df.show()


df1 =df.collect()
print(df1)

print(df1[0])
print(df[0][1])

## 6. DataFrame.transform()

We create custom functions with DataFrame.transform() for three main reasons in real-world data engineering: reusability, clean code (avoiding method-chain breaking), and modular testing.

In [0]:
data = [(1, 'Bhupesh' , 23) , (2, 'Shivani' ,24)]
schema = ['id', 'name', 'age']

df =spark.createDataFrame(data,schema)
df.show()

In [0]:
from pyspark.sql.functions import upper

def convertNametoUpper(df):
    return df.withColumn('name', upper(df.name))

def doubleSalary(df):
    return df.withColumn('age', df.age * 2)


In [0]:
df1 = df.transform(convertNametoUpper)
df1.show()


df2 = df.transform(doubleSalary)
df2.show()

## 7. pyspark.sql.functions.transform
## 

Definition: A higher-order SQL array function that applies a transformation function (such as a lambda expression) to every element within an array column, returning a new array column with the transformed elements.

In [0]:
from pyspark.sql.functions import col, transform


data = [("Bhupesh", [10, 20, 30]), ("Shivani", [100, 200, 300])]
df = spark.createDataFrame(data, ["name", "scores"])

# Multiply each element in the 'scores' array by 2
df_result = df.withColumn(
    "doubled_scores",
    transform(col("scores"), lambda x: x * 2)
)

df_result.show(truncate=False)

## 8. createOrReplaceTempView(table name) 
## 

** registers a DataFrame as a temporary SQL view in memory, allowing you to query distributed DataFrames using standard Spark SQL queries.**


### Tied to a single SparkSession

In [0]:
data = [(1, 'Bhupesh' , 23) , (2, 'Shivani' ,24)]
schema = ['id', 'name', 'age']

df =spark.createDataFrame(data,schema)
df.show()



dfq1 = df.select('id' ,'name').show()


df.createOrReplaceTempView('employee')    # here we creating a temp table name employee


### same thing we can do by using SQL

In [0]:
%sql

select id , name from employee 

## 9. createOrReplaceGlobalTempView()

registers a DataFrame as a global temporary SQL view tied to a system schema called global_temp. Unlike standard temporary views, global temporary views are shared across all active SparkSessions within the same Spark application


### Shared across all SparkSessions in the application  - mean we can acess in multiple notebook os same cluster spark session

In [0]:
data = [(1, 'Bhupesh' , 23) , (2, 'Shivani' ,24)]
schema = ['id', 'name', 'age']

df =spark.createDataFrame(data,schema)
df.show()


#  here we creating a temp table name employee
df.createOrReplaceGlobalTempView('employee')    


In [0]:
%sql
select id , name from global_temp.employee     \\ global_temp   is must to use before table name here

### spark.catalog.listTables('default')    ---> It will throe last temp table of session
### spark.catalog.listTables('global_temp')   --> return last global temp table name 

### spark.catalog.dropGlobalTempView('employee')   - - dlt temp name 

## 10. udf - User defined function and we can register our also

In [0]:
data = [('Bhupesh' , 1000 , 100) , ('Shivani' , 2000 , 200)]
schema = ['name', 'salary', 'bonus']

df =spark.createDataFrame(data,schema)
df.show()

